# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

El ejercicio quiere que filtremos libros según dos criterios:
-Rating mínimo: libros con rating de 4 o mas
-Precio máximo: libros con precio de hasta 20 dolares

Para ello, debemos obtener las columnas de: title, price, rating, availability.

Pasos a seguir:
1. Entrar a la web con requests.
2. Convertir el HTML con BeautifulSoup.
3. Encontrar los bloques de libros.
4. Extraer título, precio, rating y disponibilidad.
5. Convertir el resultado en un DataFrame.
5. Filtrar por:
rating >= 4
price <= 20

In [1]:
pip install requests beautifulsoup4


   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [ ]:
def scrape_books(min_rating, max_price):

    """
    Scrape books from books.toscrape.com
    filtering by minimum rating and maximum price.
    """

    # Validate inputs
    if not isinstance(min_rating, (int, float)):
        raise ValueError("min_rating must be numeric")

    if not isinstance(max_price, (int, float)):
        raise ValueError("max_price must be numeric")

    if min_rating < 1 or min_rating > 5:
        raise ValueError("min_rating must be between 1 and 5")

    if max_price < 0:
        raise ValueError("max_price must be positive")

    url = "http://books.toscrape.com/catalogue/page-1.html"

    books_data = []

    while url:

        response = requests.get(url)

        if response.status_code != 200:
            print("Error loading page")
            break

        soup = BeautifulSoup(response.text, "html.parser")

        books = soup.find_all(
            "article",
            class_="product_pod"
        )

        for book in books:

            # Extract data from catalogue page

            title = book.h3.a["title"]

            price_text = book.find(
                "p",
                class_="price_color"
            ).text.strip()

            price = float(
                price_text
                .replace("£", "")
                .replace("Â", "")
                .strip()
            )

            rating_text = book.find(
                "p",
                class_="star-rating"
            )["class"][1]

            rating = rating_map[rating_text]

            availability = (
                book.find(
                    "p",
                    class_="instock availability"
                )
                .text
                .strip()
            )

            # Apply filters

            if rating >= min_rating and price <= max_price:

                # Visit detail page

                detail_url = urljoin(
                    url,
                    book.h3.a["href"]
                )

                detail_response = requests.get(detail_url)

                if detail_response.status_code != 200:
                    continue

                detail_soup = BeautifulSoup(
                    detail_response.text,
                    "html.parser"
                )

                upc = detail_soup.find(
                    "th",
                    string="UPC"
                ).find_next("td").text

                genre = detail_soup.find(
                    "ul",
                    class_="breadcrumb"
                ).find_all("a")[2].text

                description_section = detail_soup.find(
                    "div",
                    id="product_description"
                )

                if description_section:
                    description = (
                        description_section
                        .find_next("p")
                        .text
                        .strip()
                    )
                else:
                    description = "No description available"

                books_data.append({
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description
                })

        # Move to next page

        next_button = soup.find(
            "li",
            class_="next"
        )

        if next_button:
            url = urljoin(
                url,
                next_button.a["href"]
            )
        else:
            url = None

    return pd.DataFrame(
        books_data,
        columns=[
            "UPC",
            "Title",
            "Price (£)",
            "Rating",
            "Genre",
            "Availability",
            "Description"
        ]
    )

In [ ]:
books_df = scrape_books(
    min_rating=4,
    max_price=20
)

In [ ]:
books_df.head()

In [ ]:
books_df.columns

In [ ]:
books_df.shape